# Preparando os dados

1. Importo as bibliotecas necessárias para todos os processos que farei:

In [1]:
import pandas as pd
from sklearn.feature_selection import SelectKBest
from sklearn.feature_selection import f_classif, mutual_info_classif
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

2. Peço para que o algoritmo leia o database:

In [2]:
df = pd.read_excel('csgo_round_snapshots.xlsx')

3. Utilizo o atributo `shape` para saber quantas colunas (*features*) tem no database:

In [3]:
df.shape # para saber quantas features

(122410, 94)

4. Utilizo o atributo info para saber o datatype de cada coluna e verificar se alguma delas pode criar ruído nas análises:

In [4]:
df.info() # para conferir o data type das features

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 122410 entries, 0 to 122409
Data columns (total 94 columns):
 #   Column                        Non-Null Count   Dtype  
---  ------                        --------------   -----  
 0   time_left                     122410 non-null  float64
 1   ct_score                      122410 non-null  int64  
 2   t_score                       122410 non-null  int64  
 3   map                           122410 non-null  object 
 4   ct_health                     122410 non-null  int64  
 5   t_health                      122410 non-null  int64  
 6   ct_money                      122410 non-null  int64  
 7   t_money                       122410 non-null  int64  
 8   ct_helmets                    122410 non-null  int64  
 9   t_helmets                     122410 non-null  int64  
 10  ct_defuse_kits                122410 non-null  int64  
 11  ct_players_alive              122410 non-null  int64  
 12  t_players_alive               122410 non-nul

Concluo que a coluna `‘map’` pode me atrapalhar na seleção das melhores features para o treinamento do algoritmo.
Além disso, temos nossa variável alvo (target) `‘round_winner’`, a que vai servir de base para conferir o quão próxima a previsão vai estar da realidade.

5. Separo as features do target e excluo a coluna ‘map’:

In [5]:
X = df.drop(columns = ['round_winner', 'map'])
y = df['round_winner']

> X ser maiúsculo e y ser minúsculo em exemplos e projetos tem uma razão!
É uma convenção matemática/computacional, já que X é usado para representar características (matriz), e y o alvo (vetor).

6. O código a seguir nos mostra colunas em que os valores são constantes, ou seja, não são úteis para mostrar a diferença entre as equipes. As identificamos com o método `nunique()` do Pandas, e as excluímos:

In [6]:
constant_cols = X.columns[X.nunique() <= 1]
X[constant_cols].head()

,ct_weapon_bizon,ct_weapon_g3sg1,t_weapon_m249,ct_weapon_negev,ct_weapon_r8revolver,ct_weapon_sawedoff
0,0,0,0,0,0,0
1,0,0,0,0,0,0
2,0,0,0,0,0,0
3,0,0,0,0,0,0
4,0,0,0,0,0,0


In [7]:
X = X.drop(columns = constant_cols)

# Divisão do dataset

1. Utilizo a função `train_test_split( )` para dividir os dados em conjuntos de treinamento e teste. Destino 25% a teste e o restante para o treinamento. É importante para que eu consiga avaliar se o modelo funciona com dados que ele não usou no treinamento:

In [8]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size = 0.25, random_state = 2000)

2. Uso `head()` para rapidamente conferir se a separação está correta visualmente:

In [9]:
X_train.head()

,time_left,ct_score,t_score,ct_health,t_health,ct_money,t_money,ct_helmets,t_helmets,ct_defuse_kits,...,ct_grenade_flashbang,t_grenade_flashbang,ct_grenade_smokegrenade,t_grenade_smokegrenade,ct_grenade_incendiarygrenade,t_grenade_incendiarygrenade,ct_grenade_molotovgrenade,t_grenade_molotovgrenade,ct_grenade_decoygrenade,t_grenade_decoygrenade
66505,54.89,1,8,492,500,250,58000,0,5,0,...,4,5,3,3,1,0,0,3,0,0
108375,175.00,8,0,500,500,59700,18200,3,0,3,...,4,0,4,0,4,0,0,0,0,0
27174,14.90,11,10,371,170,37300,600,3,2,4,...,3,1,2,0,0,0,0,1,0,0
57525,169.91,5,4,500,500,22650,21100,0,2,0,...,0,2,0,0,0,0,0,0,0,0
73383,106.91,5,13,500,500,10500,500,4,2,3,...,5,1,5,0,5,0,0,0,0,0


In [10]:
X_test.head()

,time_left,ct_score,t_score,ct_health,t_health,ct_money,t_money,ct_helmets,t_helmets,ct_defuse_kits,...,ct_grenade_flashbang,t_grenade_flashbang,ct_grenade_smokegrenade,t_grenade_smokegrenade,ct_grenade_incendiarygrenade,t_grenade_incendiarygrenade,ct_grenade_molotovgrenade,t_grenade_molotovgrenade,ct_grenade_decoygrenade,t_grenade_decoygrenade
77680,69.94,1,3,467,497,12850,2700,3,5,3,...,5,4,3,3,3,0,0,3,0,0
30145,174.94,6,11,500,500,9500,18700,0,1,0,...,0,0,0,0,0,0,0,0,0,0
94364,159.19,9,15,500,500,1350,4200,2,5,2,...,2,5,4,5,3,0,0,5,0,0
7026,74.96,10,13,399,390,5050,8950,4,4,2,...,4,4,1,2,3,0,0,4,0,0
101522,54.91,12,5,109,353,6100,12600,0,4,0,...,0,3,0,0,0,0,0,1,0,0


## Seleção de atributos

1. Inicializo o `SelectKBest`, definindo que serão selecionados os 10 atributos mais relevantes para prever a variável-alvo, utilizando a função `f_classif` para avaliar cada atributo.

In [11]:
seletor = SelectKBest(score_func = f_classif, k = 10)

2. Em seguida, utilizo o método `fit_transform()` para identificar e selecionar os 10 melhores atributos do conjunto de treinamento. Depois, aplico o método `transform()` ao conjunto de teste, mantendo apenas os mesmos atributos selecionados anteriormente.

In [12]:
X_train_sel = seletor.fit_transform(X_train, y_train)
X_test_sel = seletor.transform(X_test)

3. Por fim, utilizo o método `get_support()` para visualizar as colunas selecionadas.

In [13]:
cols = seletor.get_support(indices = True)
X_train.iloc[:, cols]

,ct_health,ct_helmets,t_helmets,ct_defuse_kits,ct_players_alive,t_weapon_ak47,ct_weapon_awp,ct_weapon_m4a4,ct_grenade_flashbang,ct_grenade_smokegrenade
66505,492,0,5,0,5,3,1,2,4,3
108375,500,3,0,3,5,0,2,0,4,4
27174,371,3,2,4,4,0,2,0,3,2
57525,500,0,2,0,5,1,0,0,0,0
73383,500,4,2,3,5,0,0,0,5,5
...,...,...,...,...,...,...,...,...,...,...
34358,500,0,1,0,5,0,0,0,0,0
69581,100,1,3,1,1,2,0,0,1,1
62670,500,0,5,0,5,2,1,2,3,3
37704,496,4,5,3,5,3,1,4,4,2


Aqui finaliza a seleção de atributos.

# Treinamento

1. Crio o modelo de árvore de decisão. Em seguida, utilizo o método `fit()` para treinar o modelo com os atributos selecionados e seus respectivos resultados reais:

In [14]:
modelo_dtc = DecisionTreeClassifier(random_state = 2000)
modelo_dtc.fit(X_train_sel, y_train)

,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness of the estimator. The features are alwaysrandomly permuted at each split, even if ``splitter`` is set to``""best""``. When ``max_features < n_features``, the algorithm willselect ``max_features`` at random at each split before finding the bestsplit among them. But the best found split may vary across differentruns, even if ``max_features=n_features``. That is the case, if theimprovement of the criterion is identical for several splits and onesplit has to be selected at random. To obtain a deterministic behaviourduring fitting, ``random_state`` has to be fixed to an integer.See :term:`Glossary <random_state>` for details.",2000
,"criterion criterion: {""gini"", ""entropy"", ""log_loss""}, default=""gini""The function to measure the quality of a split. Supported criteria are""gini"" for the Gini impurity and ""log_loss"" and ""entropy"" both for theShannon information gain, see :ref:`tree_mathematical_formulation`.",'gini'
,"splitter splitter: {""best"", ""random""}, default=""best""The strategy used to choose the split at each node. Supportedstrategies are ""best"" to choose the best split and ""random"" to choosethe best random split.",'best'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: int, float or {""sqrt"", ""log2""}, default=NoneThe number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None, then `max_features=n_features`... note:: The search for a split does not stop until at least one valid partition of the node samples is found, even if it requires to effectively inspect more than ``max_features`` features.",None
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow a tree with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.The weighted impurity decrease equation is the following:: N_t / N * (impurity - N_t_R / N_t * right_impurity - N_t_L / N_t * left_impurity)where ``N`` is the total number of samples, ``N_t`` is the number ofsample

2. Realizo as previsões com o modelo treinado e exibo as 10 primeiras:

In [15]:
y_predict = modelo_dtc.predict(X_test_sel)
print(y_predict[:10])

['T' 'T' 'T' 'CT' 'T' 'CT' 'T' 'CT' 'T' 'T']


Comparação de resultados:

In [16]:
print(pd.DataFrame({
    'Real': y_test,
    '| DTC': y_predict 
}).head(10))

       Real | DTC
77680     T     T
30145     T     T
94364     T     T
7026     CT    CT
101522    T     T
50029    CT    CT
41732    CT     T
43972    CT    CT
17367     T     T
66447     T     T


3. Utilizo a função `accuracy_score()` para comparar as previsões do modelo com os resultados reais. A acurácia obtida foi de aproximadamente 73,41%, ou seja, o modelo acertou cerca de 73 de cada 100 previsões realizadas no conjunto de teste.

In [17]:
acy = accuracy_score(y_test, y_predict)
print('Acurácia do modelo:', acy)

Acurácia do modelo: 0.7341110348658628
